# Director and Cast Consistency Analysis

## Q1. Director–Genre Frequency

- Why Critical: High-cardinality GroupBy
    
- Metric: Aggregation time

    
## Q2. Director Success Rate by Genre

- Why Critical: Multi-stage aggregation

- Metric: GroupBy + aggregation cost

    
## Q3. Cast Genre Specialization

- Why Critical: Pattern detection

- Metric: Filter + GroupBy cost

    
## Q4. Director Versatility Score

- Why Critical: Distinct-count operation

- Metric: Cardinality computation cost

    
## Q5. Recurring Director–Cast Combinations

- Why Critical: Multi-way join

    
- Metric: Join complexity


In [1]:
# Import Libraries
import pandas as pd
import numpy as np
import time
import tracemalloc
import ast

# Start Total Timer
start_total = time.perf_counter()
tracemalloc.start()

# Load Data
movies = pd.read_csv(
    "movies_cleaned.csv",
    dtype={
        "vote_average": "float32",
        "vote_count": "int32"
    }
)

print("Initial Shape:", movies.shape)
print("Initial Memory (MB):", round(movies.memory_usage(deep=True).sum()/1024**2,4))


# PREPARATION
movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")
movies["year"] = movies["release_date"].dt.year.astype("int16")

def parse_list(x):
    if isinstance(x, str):
        try:
            if x.startswith("["):
                return ast.literal_eval(x)
            return [i.strip() for i in x.split(",")]
        except:
            return ["Unknown"]
    return x

movies["genres"] = movies["genres"].apply(parse_list)
movies["cast"] = movies["cast"].apply(parse_list)
movies["director"] = movies["director"].apply(parse_list)


# Q1 — Director–Genre Frequency
start = time.perf_counter()

dg = movies.explode("director").explode("genres")

director_genre_freq = (
    dg.groupby(["director","genres"])
      .size()
      .reset_index(name="count")
)

end = time.perf_counter()
q1_time = end - start

print("\nQ1: Director–Genre Frequency")
print("Aggregation Time (sec):", round(q1_time,4))


# Q2 — Director Success Rate by Genre
start = time.perf_counter()

director_success = (
    dg.groupby(["director","genres"])
      .agg(
          avg_rating=("vote_average","mean"),
          total_movies=("id","count")
      )
      .reset_index()
)

end = time.perf_counter()
q2_time = end - start

print("\nQ2: Director Success Rate by Genre")
print("Multi-stage Aggregation Time (sec):", round(q2_time,4))


# Q3 — Cast Genre Specialization
# Explode ONLY cast & genre
start = time.perf_counter()

cg = movies.explode("cast").explode("genres")

cast_genre = (
    cg.groupby(["cast","genres"])
      .size()
      .reset_index(name="count")
)

specialized_cast = cast_genre[cast_genre["count"] > 5]

end = time.perf_counter()
q3_time = end - start

print("\nQ3: Cast Genre Specialization")
print("Filter + GroupBy Time (sec):", round(q3_time,4))


# Q4 — Director Versatility Score
start = time.perf_counter()

director_versatility = (
    dg.groupby("director")["genres"]
      .nunique()
      .reset_index(name="versatility_score")
)

end = time.perf_counter()
q4_time = end - start

print("\nQ4: Director Versatility Score")
print("Distinct Count Time (sec):", round(q4_time,4))


# Q5 — Recurring Director–Cast Combinations
# Explode director & cast ONLY (no genres)
start = time.perf_counter()

dc = movies.explode("director").explode("cast")

director_cast_pairs = (
    dc.groupby(["director","cast"])
      .size()
      .reset_index(name="collaboration_count")
)

recurring_pairs = director_cast_pairs[
    director_cast_pairs["collaboration_count"] > 2
]

end = time.perf_counter()
q5_time = end - start

print("\nQ5: Recurring Director–Cast Combinations")
print("Join Complexity Time (sec):", round(q5_time,4))


# Total Execution Time
end_total = time.perf_counter()
total_time = end_total - start_total

current, peak = tracemalloc.get_traced_memory()

print("Total Execution Time (sec):", round(total_time,4))
print("Peak Memory Usage (MB):", round(peak/1024**2,4))

tracemalloc.stop()

Initial Shape: (590202, 166)
Initial Memory (MB): 2345.9759

Q1: Director–Genre Frequency
Aggregation Time (sec): 19.159

Q2: Director Success Rate by Genre
Multi-stage Aggregation Time (sec): 2.5668


MemoryError: Unable to allocate 21.9 MiB for an array with shape (11491239, 1) and data type int16